In [4]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

df = pd.read_csv("../data/top1_wealth_share_uk.csv")
df["year"] = pd.to_datetime(df["year"], format="%Y")

ref = pd.DataFrame({"year": pd.to_datetime([1926], format="%Y")})

styles = EcoStyles()
styles.register_and_enable_theme()

# hover -> snap to nearest year, show guide rule + point + tooltip
hover = alt.selection_point(fields=["year"], nearest=True, on="pointerover",
                            empty=False, clear="pointerout")

line = alt.Chart(df).mark_line(strokeWidth=1.8, color="#122B39").encode(
    x=alt.X("year:T", title=None, axis=alt.Axis(format="%Y", tickCount=10)),
    y=alt.Y("share:Q", title="Top 1% wealth share (%)", scale=alt.Scale(domain=[0, 80])),
)

rule = alt.Chart(ref).mark_rule(color="#E6224B", strokeWidth=1).encode(x="year:T")
label = alt.Chart(ref).mark_text(color="#E6224B", align="left", baseline="top",
                                 dx=5, dy=5).encode(
    x="year:T", y=alt.value(0), text=alt.value("1926"))

hover_rule = alt.Chart(df).mark_rule(color="#94a3b8", strokeWidth=1).encode(
    x="year:T",
    opacity=alt.condition(hover, alt.value(0.5), alt.value(0)),
)
hover_point = alt.Chart(df).mark_point(size=70, filled=True, color="#122B39").encode(
    x="year:T", y="share:Q",
    opacity=alt.condition(hover, alt.value(1), alt.value(0)),
    tooltip=[alt.Tooltip("year:T",  title="Year", format="%Y"),
             alt.Tooltip("share:Q", title="Top 1% share", format=".1f")],
).add_params(hover)

core = (
    alt.layer(hover_rule, line, rule, label, hover_point)
    .properties(width=640, height=340)
)
chart = alt.vconcat(core)          # wrap so the embed keeps the 640px size

chart.save("top1_wealth_share.json")
chart

alt.VConcatChart(...)